In [5]:
# imports
import pandas as pd
import requests
import numpy as np
import json

from data_ingestor_github import SkillCornerDataIngestor
di = SkillCornerDataIngestor()

In [6]:
# # To load all the data
# data = di.load_data(match_id=1886347)

In [7]:
# # To view the loaded data
# with open (f"../../data/gold_tracking_data.json", "r") as f:
#     gold_tracking_data = json.load(f)

# frames = gold_tracking_data['frames']

In [5]:
# start=1000
# end=1050
# filtered_frames = {frame_num: frames[str(frame_num)] for frame_num in range(start, end + 1) if str(frame_num) in frames}
# filtered_frames

In [8]:
# To view tracking data at different stages of the pipeline
bronze_tracking_data = di._get_bronze_tracking_data(match_id=1899585)
silver_tracking_data = di._get_silver_tracking_data(bronze_tracking_data)
bronze_meta_data = di._get_bronze_meta_data(match_id=1899585)
silver_meta_data = di._get_silver_meta_data(bronze_meta_data)
bronze_event_data = di._get_bronze_event_data(match_id=1899585)
silver_event_data = di._get_silver_event_data(bronze_event_data)
# gold_data = di._get_gold_tracking_data(silver_tracking_data, silver_meta_data, silver_event_data)

/Users/mihirthalanki/Documents/MT/Hackathon/PlayTactix/backend/services/data_ingestor_github.py:220: DtypeWarning: Columns (276) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_data = pd.read_csv(event_data_github_url)


In [9]:
silver_event_data.head()

,event_id,index,frame_start,frame_end,attacking_side,event_type_id,event_type,player_id,player_name,team_id,x_start,y_start,x_end,y_end
0,8_0,0,12,12,left_to_right,8,player_possession,23909,K. Barbarouses,867,0.53,-0.28,0.53,-0.28
1,8_1,1,31,44,left_to_right,8,player_possession,42510,S. Wootton,867,-20.68,0.53,-20.75,-0.04
2,7_0,2,31,32,left_to_right,7,passing_option,51015,A. Rufer,867,-9.38,-3.50,-9.36,-3.52
3,7_1,3,31,44,left_to_right,7,passing_option,799092,M. Sheridan,867,-17.97,-13.76,-18.41,-14.35
4,9_0,4,37,44,right_to_left,9,on_ball_engagement,38673,G. May,4177,11.77,-3.34,15.46,-1.41


In [10]:
silver_event_data[silver_event_data['event_type'] == 'off_ball_run'].head()

,event_id,index,frame_start,frame_end,attacking_side,event_type_id,event_type,player_id,player_name,team_id,x_start,y_start,x_end,y_end
8,1_0,8,57,66,left_to_right,1,off_ball_run,51015,A. Rufer,867,-9.98,0.48,-10.80,4.37
18,1_1,18,88,105,left_to_right,1,off_ball_run,26969,H. Ishige,867,13.18,19.94,20.60,23.58
21,1_2,21,117,125,left_to_right,1,off_ball_run,26969,H. Ishige,867,21.70,27.53,19.77,30.39
24,1_3,24,126,163,left_to_right,1,off_ball_run,27003,K. Nagasawa,867,6.12,15.14,23.88,28.47
38,1_4,38,195,227,left_to_right,1,off_ball_run,42510,S. Wootton,867,-17.83,7.35,-35.41,0.91


In [ ]:
event_idx = 0
n_events = len(silver_event_data)
events = 
while event_idx < n_events and silver_event_data.iloc[event_idx]['frame_start'] <= frame_number:
    event = silver_event_data.iloc[event_idx]
    if event['frame_start'] <= frame_number <= event['frame_end']:
        frames[frame_number]['events'].append(event.to_dict())
                if event['frame_end'] < frame_number:
                    event_idx += 1
                else:
                    break

adding event player_possession for frame 12
adding event player_possession for frame 31
adding event passing_option for frame 31
adding event passing_option for frame 31
adding event on_ball_engagement for frame 37


In [21]:
event_set = set()
for event in events:
    event_set.add(event['event_type'])
print(event_set)

{'on_ball_engagement', 'passing_option', 'player_possession'}


In [7]:
silver_data = silver_tracking_data.merge(
            silver_meta_data, left_on=["player_id"], right_on=["id"]
        )

In [8]:
pd.set_option('display.max_columns', None)  # Show all columns


In [9]:
def build_frame_dict(tracking_data, event_data):
    frames = {}
    event_idx = 0
    n_events = len(event_data)
    for frame_number, group in tracking_data.groupby("frame"):
        # Add tracking data
        frames[frame_number] = {
                #'is_detected': group['is_detected'].iloc[0],
                #'timestamp': group['timestamp'].iloc[0],
                'period': group['period'].iloc[0],
                'players': {
                    'x': group['x'].tolist(),
                    'y': group['y'].tolist(),
                    'player_id': group['player_id'].tolist(),
                    'id': group['id'].tolist(),
                    'short_name': group['short_name'].tolist(),
                    'number': group['number'].tolist(),
                    'team_id': group['team_id'].tolist(),
                    'total_time': group['total_time'].tolist(),
                    'player_role.name': group['player_role.name'].tolist(),
                    'player_role.acronym': group['player_role.acronym'].tolist(),
                    'is_gk': group['is_gk'].tolist(),
                    'direction_player_1st_half': group['direction_player_1st_half'].tolist(),
                    'direction_player_2nd_half': group['direction_player_2nd_half'].tolist(),
                },
                'ball': {
                    'ball_x': group['ball_x'].iloc[0],
                    'ball_y': group['ball_y'].iloc[0],
                    'ball_z': group['ball_z'].iloc[0],
                    #'is_detected_ball': group['is_detected_ball'].iloc[0],
                },
                'events': []
            }
        # Add event data
        while event_idx < n_events and event_data.iloc[event_idx]['frame_start'] <= frame_number:
            event = event_data.iloc[event_idx]
            if event['frame_start'] <= frame_number <= event['frame_end']:
                frames[frame_number]['events'].append(event.to_dict())
            if event['frame_end'] < frame_number:
                event_idx += 1
            else:
                break
    
    return frames


In [10]:
gold_frames = build_frame_dict(silver_data, silver_event_data)

KeyboardInterrupt: 

In [ ]:
gold_frames[58]['events']

[{'event_id': '8_1',
  'index': 1,
  'frame_start': 48,
  'frame_end': 58,
  'attacking_side': 'left_to_right',
  'event_type_id': 8,
  'event_type': 'player_possession',
  'event_subtype_id': nan,
  'event_subtype': nan,
  'player_id': 51649,
  'player_name': 'A. Šušnjar',
  'team_id': 1805,
  'x_start': -22.31,
  'y_start': 1.22,
  'x_end': -22.21,
  'y_end': 2.7}]

In [ ]:
silver_event_data.head()

,event_id,index,frame_start,frame_end,attacking_side,event_type_id,event_type,event_subtype_id,event_subtype,player_id,player_name,team_id,x_start,y_start,x_end,y_end
0,8_0,0,28,28,left_to_right,8,player_possession,NaN,NaN,966120,B. Gibson,1805,0.73,0.49,0.73,0.49
1,8_1,1,48,58,left_to_right,8,player_possession,NaN,NaN,51649,A. Šušnjar,1805,-22.31,1.22,-22.21,2.70
2,7_0,2,48,53,left_to_right,7,passing_option,NaN,NaN,735574,K. Grozos,1805,-10.47,-2.78,-11.36,-1.38
3,7_1,3,48,58,left_to_right,7,passing_option,NaN,NaN,735578,M. Natta,1805,-20.69,16.66,-20.36,17.69
4,9_0,4,56,58,right_to_left,9,on_ball_engagement,11.0,pressing,50951,J. Brimmer,4177,13.03,0.06,13.98,-0.68
